In [ ]:
from flask import Flask, jsonify, request, render_template
from flask_cors import CORS
from werkzeug.serving import run_simple
from datetime import datetime
import requests

app = Flask(__name__)
CORS(app)

payments = [
    {"payment_id": "P001", "customer_id": "C001", "item_id": ["I001", "I002"], "qty": [30, 40], "amount": 65000,"address": "pyay","phno": "09683969002","method": "payment", "status": "COMFIRM", "date": "2025-08-01"},
    {"payment_id": "P002", "customer_id": "C001", "item_id": ["I003", "I004"], "qty": [16, 17], "amount": 50000,"address": "pyay","phno": "09683969002", "method": "Loan", "status": "COMFIRM", "date": "2025-08-02"},
    {"payment_id": "P003", "customer_id": "C002", "item_id": ["I011", "I010"], "qty": [10, 20], "amount": 53000,"address": "pyay","phno": "09683969002", "method": "payment","status": "Order","date": "2025-08-02"},
    {"payment_id": "P004", "customer_id": "C002","item_id": ["I014", "I020"], "qty": [30, 40], "amount": 120000,"address": "pyay","phno": "09683969002", "method": "Loan", "status": "Order", "date": "2025-08-02"}
]

loans = [
    {"loan_id": "L001", "payment_id": "P002", "name": "Kevin", "age": 30, "annual_income": 300000, "marital_status": "single", "education": "Bachelor", "credit_score": 750},
    {"loan_id": "L002", "payment_id": "P004", "name": "Marshal", "age": 29, "annual_income": 250000, "marital_status": "married", "education": "Master", "credit_score": 500}
]

@app.route('/')
def index():
    return render_template("userView.html")

@app.route('/userView')
def userView():
    return render_template('userView.html')

@app.route('/payment')
def payment():
    return render_template('payment.html')

@app.route('/proceedPayment', methods=['POST'])
def proceed_payment():
    data = request.get_json()
    if not data:
        return jsonify({"error": "Invalid or missing JSON"}), 400
    
    customer_id = data.get('customer_id')
    cart = data.get('cart')  
    address = data.get('address')
    phno = data.get('phno')
    method = data.get('method', 'payment')
    status = data.get('status', 'Order')
    today_str = datetime.now().strftime("%Y-%m-%d")

    if not (customer_id and cart and address and phno):
        return jsonify({"error": "Missing fields"}), 400

    item_ids = [item.get('itemId') for item in cart]
    qtys = [item.get('qtyInCart') for item in cart]

    total_amount = 0
    for item in cart:
        qty = item.get('qtyInCart', 0)
        price = item.get('price', 0)
        total_amount += qty * price

    new_payment_id = "P" + str(len(payments) + 1).zfill(3)

    new_payment = {
        "payment_id": new_payment_id,
        "customer_id": customer_id,
        "item_id": item_ids,
        "qty": qtys,
        "amount": total_amount,
        "address": address,
        "phno": phno,
        "method": method,
        "status": status,
        "date": today_str
    }
    
    payments.append(new_payment)

    # Forward to Server 1
    try:
        response = requests.post("http://192.168.181.241:4003/newPayment", json=new_payment)
        response.raise_for_status()
    except Exception as e:
        return jsonify({"error": f"Failed to forward payment to Server 1: {str(e)}"}), 500

    return jsonify({"message": "Payment recorded and forwarded", "payment": new_payment}), 200


@app.route('/proceedloan', methods=['POST'])
def proceed_loan():
    data = request.get_json()
    if not data:
        return jsonify({"error": "Invalid or missing JSON"}), 400

    payment_id = data.get('payment_id')
    name = data.get('name')
    age = data.get('age')
    annual_income = data.get('annual_income')
    marital_status = data.get('marital_status')
    education = data.get('education')
    credit_score = data.get('credit_score')

    if not all([payment_id, name, age, annual_income, marital_status, education, credit_score]):
        return jsonify({"error": "Missing loan fields"}), 400

    new_loan_id = "L" + str(len(loans) + 1).zfill(3)
    new_loan = {
        "loan_id": new_loan_id,
        "payment_id": payment_id,
        "name": name,
        "age": age,
        "annual_income": annual_income,
        "marital_status": marital_status,
        "education": education,
        "credit_score": credit_score
    }
    
    loans.append(new_loan)

    # Forward to Server 1
    try:
        response = requests.post("http://192.168.181.241:4003/newLoans", json=new_loan)
        response.raise_for_status()
    except Exception as e:
        return jsonify({"error": f"Failed to forward loan to Server 1: {str(e)}"}), 500

    return jsonify({"message": "Loan recorded and forwarded", "loan": new_loan}), 200



# Run the app
if __name__ == '__main__':
    run_simple("192.168.181.241", 4004, app, use_reloader=False)


 * Running on http://192.168.181.241:4004
Press CTRL+C to quit
192.168.181.241 - - [12/Sep/2025 15:04:10] "GET / HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:10] "GET /favicon.ico HTTP/1.1" 404 -
192.168.181.16 - - [12/Sep/2025 15:05:29] "GET /api/supplies HTTP/1.1" 404 -
192.168.181.16 - - [12/Sep/2025 15:05:50] "GET /api/supplies HTTP/1.1" 404 -
192.168.181.16 - - [12/Sep/2025 15:21:05] "GET /api/supplies HTTP/1.1" 404 -
192.168.181.21 - - [12/Sep/2025 15:23:13] "GET / HTTP/1.1" 200 -
192.168.181.21 - - [12/Sep/2025 15:23:14] "GET /favicon.ico HTTP/1.1" 404 -
192.168.181.21 - - [12/Sep/2025 15:25:40] "GET /payment HTTP/1.1" 200 -
192.168.181.16 - - [12/Sep/2025 15:25:40] "GET /api/supplies HTTP/1.1" 404 -
